<a href="https://colab.research.google.com/github/1kaiser/opyf_colab/blob/main/jax_3d_reconstruction_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JAX 3D Reconstruction Pipeline (Zonal)
🚀 This notebook provides a high-performance JAX ecosystem for metric 3D reconstruction.

All code and datasets are now integrated directly into the **opyf_colab** repository.

## 1. Environment Setup

In [1]:
# @title 1.1 Setup & Dependencies
import sys
import os

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()

if IN_COLAB:
    print("Running in Google Colab")
    os.system("pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html")
    os.system("pip install -q flax open3d tqdm opencv-python")

    if not os.path.exists("/content/opyf_colab"):
        os.system("git clone https://github.com/1kaiser/opyf_colab.git /content/opyf_colab")

    sys.path.append("/content/opyf_colab")
    sys.path.append("/content/opyf_colab/models/jax")
    os.chdir("/content/opyf_colab")
else:
    print("Running Locally")
    project_root = os.getcwd()
    if project_root not in sys.path:
        sys.path.append(project_root)
    models_path = os.path.join(project_root, "models/jax")
    if models_path not in sys.path:
        sys.path.append(models_path)

import jax
import flax
import open3d as o3d
from tqdm import tqdm
import cv2
print(f"JAX Devices: {jax.devices()}")


Running in Google Colab
JAX Devices: [CudaDevice(id=0)]


In [4]:
# @title 1.2 Weights Management
import os

def download_weights():
    os.makedirs("weights", exist_ok=True)
    BASE_URL = "https://github.com/1kaiser/d_jax/releases/download/v1.0.0"

    print("Downloading/Verifying weights...")

    models = {
        "depth_pro.msgpack": None,
        "superpoint.msgpack": None,
        "superpoint_lightglue.msgpack": None
    }

    for target, parts in models.items():
        target_path = os.path.join("weights", target)
        if os.path.exists(target_path):
            print(f"✅ {target} already exists.")
            continue

        print(f"Downloading {target}...")
        url = f"{BASE_URL}/{target}"
        if IN_COLAB:
            os.system(f"wget -q --show-progress {url} -O {target_path}")
        else:
            import subprocess
            subprocess.run(["wget", "-q", "--show-progress", url, "-O", target_path])

    print("\n✅ Required weights ready.")

if IN_COLAB:
    download_weights()
else:
    required = ["depth_pro.msgpack", "superpoint.msgpack", "superpoint_lightglue.msgpack"]
    missing = [r for r in required if not os.path.exists(os.path.join("weights", r))]
    if missing:
        print(f"Missing weights: {missing}. Downloading...")
        download_weights()
    else:
        print("✅ Local weights detected.")


Downloading/Verifying weights...
✅ depth_pro.msgpack already exists.
✅ superpoint.msgpack already exists.
✅ superpoint_lightglue.msgpack already exists.

✅ Required weights ready.


## 2. Standalone Model Testing

In [5]:
if IN_COLAB:   test_img1 = "./data/pinecone_subset/frame_001.jpg"
else:
     test_img1 = "./data/pinecone_subset/frame_001.jpg"
os.makedirs("output", exist_ok=True)
os.system(f"python -m inference.infer_depth_pro --image {test_img1} --weights weights/depth_pro.msgpack --output output/depth_result.jpg")

256

## 3. Zonal 3D Reconstruction

In [13]:
!wget -O /content/opyf_colab/weights/depth_pro.msgpack https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/depth_pro.msgpack
!wget -O /content/opyf_colab/weights/superpoint.msgpack https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/superpoint.msgpack
!wget -O /content/opyf_colab/weights/superpoint_lightglue.msgpack https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/superpoint_lightglue.msgpack



--2026-03-11 03:49:24--  https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/depth_pro.msgpack
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/688344480/683826b7-ad91-49e3-9771-2661cceb796a?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-11T04%3A44%3A38Z&rscd=attachment%3B+filename%3Ddepth_pro.msgpack&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-11T03%3A44%3A33Z&ske=2026-03-11T04%3A44%3A38Z&sks=b&skv=2018-11-09&sig=JdAOhLSjmlRyjRmD0a7XZZc5EQe%2FLprRw5NJjnXSkBM%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MzIwNDU2NCwibmJmIjoxNzczMjAwOTY0LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjd

In [ ]:
from pipelines.pipeline_jax import ReconstructionPipeline

pipeline = ReconstructionPipeline("weights/depth_pro.msgpack", "weights/superpoint.msgpack", "weights/superpoint_lightglue.msgpack")

T_globals = pipeline.run("./data/pinecone_subset", output_path="output/pinecone_reconstruction", num_zones=3, radial_clip=0.7)

Loading weights from weights/depth_pro.msgpack, weights/superpoint.msgpack, weights/superpoint_lightglue.msgpack...
Loading Depth Pro weights...
Loading SuperPoint weights...
Loading LightGlue weights...
Compiling JIT functions (this may take a few minutes)...
Pipeline initialized.


Reconstructing (3 Zones, Clip=0.7):   0%|          | 0/9 [00:00<?, ?it/s]

## 4. Visualization

In [ ]:
#################live glb viewer#################
from IPython.display import HTML
import base64

# Load GLB as base64
with open("/content/model.glb", "rb") as f:
    glb_data = base64.b64encode(f.read()).decode("utf-8")

HTML(f"""
<iframe srcdoc="
<!DOCTYPE html>
<html lang='en'>
  <head>
    <script type='module' src='https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js'></script>
    <style>html, body {{ margin: 0; height: 100%; }}</style>
  </head>
  <body>
    <model-viewer
      src='data:model/gltf-binary;base64,{glb_data}'
      alt='3D scene'
      auto-rotate
      camera-controls
      background-color='#FFFFFF'
      style='width:100%; height:100%;'>
    </model-viewer>
  </body>
</html>
" width="100%" height="600px" style="border:0;"></iframe>
""")